In [1]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import pymongo
import os
from tensorflow.keras.utils import plot_model
import gc
from tqdm import tqdm
import tensorflow as tf
import time

2024-12-10 12:41:23.298464: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-10 12:41:24.184327: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
def create_new_version_dir(base_path):
    version = 1
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            os.makedirs(version_dir)
            print(f"Created new directory: {version_dir}")
            return version_dir
        version += 1

In [3]:
# Funktion zur Überwachung des Speicherverbrauchs
def check_memory_usage(threshold=90):
    memory = psutil.virtual_memory()
    if memory.percent > threshold:
        print(f"Warnung: Speicherverbrauch bei {memory.percent}%. Programm wird gestoppt.")
        raise MemoryError("Speicherverbrauch zu hoch")

In [4]:
# Load and Merge Data (Assuming MongoDB setup is correct)
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                         left_on='fingerprintId', 
                         right_on='_id', 
                         suffixes=('_sample', '_fingerprint'))
    
    return merged_df

merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

Gesamtdatensatz enthält 2000000 Einträge.


In [5]:
# Preprocess Data

# Function to process images in RGB
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')  # Convert image to RGB
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalize to values between 0 and 1
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

# Function to extract images and labels from DataFrame
def extract_images_from_df(df, example_user_id):
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Lade Bilder"):
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Example user ID
example_user_id = 'benutzername_1'

# Create DataFrames for each user
user_ids = merged_df['username'].unique()
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Sample data for the example user
user_df = user_dfs[example_user_id].sample(n=15000, random_state=42)

# Add negative examples and limit
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=15000, random_state=42)

# Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Process image data
try:
    X_train, y_train = extract_images_from_df(train_df, example_user_id)
    X_val, y_val = extract_images_from_df(val_df, example_user_id)
    X_test, y_test = extract_images_from_df(test_df, example_user_id)
except Exception as e:
    print(f"Fehler bei der Bilddatenverarbeitung: {e}")

# Free up memory
gc.collect()

Lade Bilder: 100%|██████████| 3000/3000 [00:02<00:00, 1352.58it/s]


0

In [6]:
# Define different CNN models

# Model 1: Simple CNN
# This model is a simple CNN with one convolutional layer followed by a max-pooling layer, 
# a flatten layer, and two dense layers. It is inspired by basic CNN architectures used for image classification.
def create_model_1(input_shape):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model 2: CNN with Batch Normalization
# This model adds batch normalization layers after each convolutional layer to normalize the activations 
# and improve training stability. It is inspired by modern CNN architectures that use batch normalization.
def create_model_2(input_shape):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model 3: Deeper CNN with Batch Normalization
# This model is a deeper CNN with three convolutional layers, each followed by batch normalization and max-pooling layers.
# It is inspired by deeper CNN architectures that use multiple convolutional layers to learn more complex features.
def create_model_3(input_shape):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model 4: Even Deeper CNN with Batch Normalization
# This model is an even deeper CNN with four convolutional layers, each followed by batch normalization and max-pooling layers.
# It is inspired by very deep CNN architectures that use multiple convolutional layers to learn very complex features.
def create_model_4(input_shape):
    model = Sequential([
        Conv2D(64, (3, 3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(256, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(512, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model 5: Very Deep CNN with Batch Normalization
# This model is a very deep CNN with five convolutional layers, each followed by batch normalization and max-pooling layers.
# It is inspired by very deep CNN architectures that use multiple convolutional layers to learn very complex features.
def create_model_5(input_shape):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(256, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(512, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# List of models to test
models = [create_model_1, create_model_2, create_model_3, create_model_4, create_model_5]

# Create directory for saving models
os.makedirs('Experimente/CNN', exist_ok=True)

# Plot model architectures and save models
for i, model_fn in enumerate(models, 1):
    model = model_fn((224, 224, 3))
    plot_model(model, to_file=f'/app/Experimente/CNN/model_{i}.png', show_shapes=True, show_layer_names=True)
    model.save(f'/app/Experimente/CNN/model_{i}.h5')
    print(f'Model architecture saved as /app/Experimente/CNN/model_{i}.png')
    print(f'Model saved as /app/Experimente/CNN/model_{i}.h5')
    model.summary()

2024-12-10 12:43:35.979643: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-12-10 12:43:36.136936: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-12-10 12:43:36.137247: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Model architecture saved as /app/Experimente/CNN/model_1.png
Model saved as /app/Experimente/CNN/model_1.h5
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 flatten (Flatten)           (None, 394272)            0         
                                                                 
 dense (Dense)               (None, 128)               50466944  
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dens

/usr/local/lib/python3.8/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model architecture saved as /app/Experimente/CNN/model_2.png
Model saved as /app/Experimente/CNN/model_2.h5
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_1 (Conv2D)           (None, 222, 222, 32)      896       
                                                                 
 batch_normalization (Batch  (None, 222, 222, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 111, 111, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 batch_normalization_1 (Bat  (None, 109, 109, 64)      256       
 chNormaliza

/usr/local/lib/python3.8/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Total params: 11169985 (42.61 MB)
Trainable params: 11169537 (42.61 MB)
Non-trainable params: 448 (1.75 KB)
_________________________________________________________________


/usr/local/lib/python3.8/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model architecture saved as /app/Experimente/CNN/model_4.png
Model saved as /app/Experimente/CNN/model_4.h5
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 222, 222, 64)      1792      
                                                                 
 batch_normalization_5 (Bat  (None, 222, 222, 64)      256       
 chNormalization)                                                
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 111, 111, 64)      0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 109, 109, 128)     73856     
                                                                 
 batch_normalization_6 (Bat  (None, 109, 109, 128)     512       
 chNormaliza

/usr/local/lib/python3.8/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [7]:
def prepare_images(df, num_samples):
    images = []
    labels = []
    sampled_df = df.sample(n=num_samples, random_state=42)
    for _, row in sampled_df.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(row['username'])
    return np.array(images), np.array(labels)

In [ ]:
# Define early stopping and reduce learning rate on plateau callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10)
reduce_lr_on_plateau = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

base_path = '/ssd'


# Function to run experiment with different models and sample sizes
def run_experiment(models, sample_sizes, negative_sample_ratios, base_path):
    results = []
    total_iterations = len(models) * len(sample_sizes) * len(negative_sample_ratios)
    version_dir = create_new_version_dir(base_path)
    with tqdm(total=total_iterations, desc="Total Progress", unit="iteration") as pbar:
        for model_fn in models:
            for sample_size in sample_sizes:
                for neg_ratio in negative_sample_ratios:
                    start_time = time.time()
                    print(f"Running experiment with model {model_fn.__name__}, sample size: {sample_size}, negative ratio: {neg_ratio}")
                    
                    # Sample data for the example user
                    user_df = user_dfs[example_user_id].sample(n=sample_size, random_state=42)
                    
                    # Add negative examples and limit
                    negative_sample_size = int(sample_size * neg_ratio)
                    negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
                    negative_df = negative_df.sample(n=negative_sample_size, random_state=42)
                    
                    # Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
                    train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
                    train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)
                    
                    # Process image data
                    try:
                        X_train, y_train = extract_images_from_df(train_df, example_user_id)
                        X_val, y_val = extract_images_from_df(val_df, example_user_id)
                        X_test, y_test = extract_images_from_df(test_df, example_user_id)
                    except Exception as e:
                        print(f"Fehler bei der Bilddatenverarbeitung: {e}")
                        continue
                    
                    # Define and train the model
                    model = model_fn((224, 224, 3))
                    model.fit(X_train, y_train,
                              validation_data=(X_val, y_val),
                              epochs=50,
                              batch_size=32,
                              callbacks=[early_stopping, reduce_lr_on_plateau])
                    
                    # Save model architecture and weights
                    model_name = f"{model_fn.__name__}_samplesize_{sample_size}_negratio_{neg_ratio}"
                    plot_model(model, to_file=os.path.join(version_dir, f'{model_name}.png'), show_shapes=True, show_layer_names=True)
                    model.save(os.path.join(version_dir, f'{model_name}.h5'))
                    print(f'Model architecture saved as {os.path.join(version_dir, f"{model_name}.png")}')
                    print(f'Model saved as {os.path.join(version_dir, f"{model_name}.h5")}')
                    model.summary()
                    
                    
                    # Evaluate the model on the test set
                    test_loss, test_accuracy = model.evaluate(X_test, y_test)
                    y_pred = model.predict(X_test)
                    y_pred_labels = (y_pred > 0.5).astype(int)
                    
                    # Calculate additional metrics
                    precision = precision_score(y_test, y_pred_labels)
                    recall = recall_score(y_test, y_pred_labels)
                    f1 = f1_score(y_test, y_pred_labels)
                    roc_auc = auc(*roc_curve(y_test, y_pred_labels)[:2])
                    
                    print(f"Test Accuracy for model {model_fn.__name__} with sample size {sample_size} and negative ratio {neg_ratio}: {test_accuracy:.4f}")
                    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, ROC AUC: {roc_auc:.4f}")
                    
                    # Save the results
                    results.append((model_fn.__name__, sample_size, neg_ratio, test_accuracy, precision, recall, f1, roc_auc))
                    
                    # Free up memory
                    del model, X_train, y_train, X_val, y_val, X_test, y_test
                    gc.collect()
                    tf.keras.backend.clear_session()
                    
                    # Update progress bar
                    elapsed_time = time.time() - start_time
                    pbar.set_postfix({"Elapsed Time": f"{elapsed_time:.2f} s"})
                    pbar.update(1)
    
    return results

# Define sample sizes and negative sample ratios for the experiment
sample_sizes = [12000, 14000, 16000, 18000, 20000]  # Number of samples to use for training
negative_sample_ratios = [0.5, 1, 2]  # Ratios of negative samples to positive samples

# Run the experiment
experiment_results = run_experiment(models, sample_sizes, negative_sample_ratios, base_path)

# Display the results
results_df = pd.DataFrame(experiment_results, columns=['Model', 'Sample Size', 'Negative Ratio', 'Test Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'])
print(results_df)

# Plot the results
plt.figure(figsize=(12, 6))
for model_name in results_df['Model'].unique():
    for neg_ratio in negative_sample_ratios:
        model_results = results_df[(results_df['Model'] == model_name) & (results_df['Negative Ratio'] == neg_ratio)]
        plt.plot(model_results['Sample Size'], model_results['Test Accuracy'], marker='o', label=f'{model_name} (Neg Ratio {neg_ratio})')
plt.title('Test Accuracy vs. Sample Size for Different Models and Negative Ratios')
plt.xlabel('Sample Size')
plt.ylabel('Test Accuracy')
plt.xscale('log')
plt.legend()
plt.grid(True)
plt.show()

# Plot additional metrics
metrics = ['Precision', 'Recall', 'F1 Score', 'ROC AUC']
for metric in metrics:
    plt.figure(figsize=(12, 6))
    for model_name in results_df['Model'].unique():
        for neg_ratio in negative_sample_ratios:
            model_results = results_df[(results_df['Model'] == model_name) & (results_df['Negative Ratio'] == neg_ratio)]
            plt.plot(model_results['Sample Size'], model_results[metric], marker='o', label=f'{model_name} (Neg Ratio {neg_ratio})')
    plt.title(f'{metric} vs. Sample Size for Different Models and Negative Ratios')
    plt.xlabel('Sample Size')
    plt.ylabel(metric)
    plt.xscale('log')
    plt.legend()
    plt.grid(True)
    plt.show()

Created new directory: /ssd/v2


Total Progress:   0%|          | 0/75 [00:00<?, ?iteration/s]

Running experiment with model create_model_1, sample size: 12000, negative ratio: 0.5


Lade Bilder: 100%|██████████| 3600/3600 [00:02<00:00, 1286.76it/s]


Total Progress:   0%|          | 0/75 [00:53<?, ?iteration/s]


KeyboardInterrupt: 

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import numpy as np

# Function to get the latest version directory
def get_latest_version_dir(base_path):
    version = 1
    latest_version_dir = None
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            break
        latest_version_dir = version_dir
        version += 1
    return latest_version_dir

In [ ]:
# Function to prepare images from the dataframe
def prepare_images(df, num_samples, example_user_id):
    images = []
    labels = []
    
    # Ensure at least 1000 samples from the example user
    example_user_samples = df[df['username'] == example_user_id].sample(n=1000, random_state=42)
    other_samples = df[df['username'] != example_user_id].sample(n=num_samples, random_state=42)
    
    sampled_df = pd.concat([example_user_samples, other_samples])
    
    for _, row in sampled_df.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(row['username'])
    
    return np.array(images), np.array(labels)

In [ ]:
# Test each model with 1000 samples from random users
num_samples = 1000
base_path = '/ssd'  # Ensure this is the correct base path
latest_version_dir = get_latest_version_dir(base_path)
model_names = ['create_model_1', 'create_model_2', 'create_model_3', 'create_model_4', 'create_model_5']
sample_sizes = [100, 200, 500, 1000, 2000, 5000, 10000]
negative_sample_ratios = [0.5, 1, 2]

results = []
example_user_id = 'benutzername_1'  # Beispiel-Benutzer-ID
batch_size = 16  # Reduce batch size to avoid memory issues

In [ ]:
import tensorflow as tf
from numba import cuda

def clear_gpu_memory():
    # Clear TensorFlow GPU memory
    tf.keras.backend.clear_session()
    # Clear Numba GPU memory for both GPUs
    cuda.select_device(0)
    cuda.close()
    cuda.select_device(1)
    cuda.close()

In [1]:
for model_name in model_names:
    for sample_size in sample_sizes:
        for neg_ratio in negative_sample_ratios:
            print(f"Testing model {model_name} with {num_samples} samples from random users")
            
            # Load the model
            model_path = os.path.join(latest_version_dir, f'{model_name}_samplesize_{sample_size}_negratio_{neg_ratio}.h5')
            print(f"Checking model path: {model_path}")
            if not os.path.exists(model_path):
                print(f"Error loading model: No file or directory found at {model_path}")
                continue
            
            try:
                model = load_model(model_path)
                print(f"Loaded model from: {model_path}")
            except Exception as e:
                print(f"Error loading model: {e}")
                continue
            
            # Prepare test data
            X_test_users, y_test_users = prepare_images(merged_df, num_samples, example_user_id)
            print(f"Test data shape: {X_test_users.shape}, Labels shape: {y_test_users.shape}")
            
            # Initialize predictions
            y_pred = []
            current_batch_size = batch_size
            
            # Ensure the model and data are on the same device
            while True:
                try:
                    with tf.device('/GPU:0'):
                        # Transfer data to GPU
                        for i in range(0, len(X_test_users), current_batch_size):
                            batch = X_test_users[i:i + current_batch_size]
                            batch = tf.convert_to_tensor(batch)
                            y_pred_batch = model.predict(batch)
                            y_pred.extend(y_pred_batch)
                    break
                except tf.errors.ResourceExhaustedError:
                    print(f"GPU memory exhausted with batch size {current_batch_size}, reducing batch size")
                    clear_gpu_memory()
                    current_batch_size = max(1, current_batch_size // 2)
                    if current_batch_size == 1:
                        print("Switching to CPU due to insufficient GPU memory")
                        with tf.device('/CPU:0'):
                            for i in range(0, len(X_test_users), current_batch_size):
                                batch = X_test_users[i:i + current_batch_size]
                                batch = tf.convert_to_tensor(batch)
                                y_pred_batch = model.predict(batch)
                                y_pred.extend(y_pred_batch)
                        break
            
            y_pred_labels = [example_user_id if pred > 0.5 else 'other_user' for pred in np.array(y_pred).flatten()]
            
            # Create a DataFrame for visualization
            results_df = pd.DataFrame({
                'Actual_User': y_test_users,
                'Predicted_User': y_pred_labels
            })
            
            # Plotting the results
            def plot_prediction_results(results_df, num_samples, example_user_id):
                plt.figure(figsize=(15, 10))
                correct_predictions = results_df[(results_df['Actual_User'] == example_user_id) & (results_df['Predicted_User'] == example_user_id)] 
                incorrect_predictions = results_df[(results_df['Actual_User'] != example_user_id) & (results_df['Predicted_User'] == example_user_id)]
                other_correct_predictions = results_df[(results_df['Actual_User'] != example_user_id) & (results_df['Predicted_User'] != example_user_id)]
                other_incorrect_predictions = results_df[(results_df['Actual_User'] == example_user_id) & (results_df['Predicted_User'] != example_user_id)]
                
                plt.scatter(correct_predictions.index, correct_predictions['Actual_User'], color='green', label='Correct Predictions (User)')
                plt.scatter(incorrect_predictions.index, incorrect_predictions['Actual_User'], color='red', label='Incorrect Predictions (User)')
                plt.scatter(other_correct_predictions.index, other_correct_predictions['Actual_User'], color='blue', label='Correct Predictions (Other)')
                plt.scatter(other_incorrect_predictions.index, other_incorrect_predictions['Actual_User'], color='orange', label='Incorrect Predictions (Other)')
                
                plt.xlabel('Sample Index')
                plt.ylabel('User')
                plt.title(f'Prediction Results for {num_samples} Samples per User')
                plt.legend()
                plt.show()

            plot_prediction_results(results_df, num_samples, example_user_id)
            
            # Confusion Matrix
            conf_matrix = confusion_matrix(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id)
            sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Other', example_user_id], yticklabels=['Other', example_user_id])
            plt.xlabel('Predicted')
            plt.ylabel('Actual')
            plt.title(f'Confusion Matrix with {num_samples} Samples per User')
            plt.show()
            
            # Accuracy
            accuracy = accuracy_score(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id)
            print(f'Accuracy: {accuracy * 100:.2f}%')
            
            # Classification Report
            print(classification_report(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id))
            
            # Calculate additional metrics
            precision = precision_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            recall = recall_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            f1 = f1_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            fpr, tpr, _ = roc_curve(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id)
            roc_auc = auc(fpr, tpr)
            
            print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, ROC AUC: {roc_auc:.4f}")
            
            # Save the results
            results.append((model_name, sample_size, neg_ratio, accuracy, precision, recall, f1, roc_auc))
            
            # Pie Chart for Prediction Accuracy
            def plot_pie_chart(results_df, example_user_id):
                total_samples = len(results_df)
                correct_predictions = len(results_df[(results_df['Actual_User'] == example_user_id) & (results_df['Predicted_User'] == example_user_id)])
                incorrect_predictions = len(results_df[(results_df['Actual_User'] != example_user_id) & (results_df['Predicted_User'] == example_user_id)])
                other_correct_predictions = len(results_df[(results_df['Actual_User'] != example_user_id) & (results_df['Predicted_User'] != example_user_id)])
                other_incorrect_predictions = len(results_df[(results_df['Actual_User'] == example_user_id) & (results_df['Predicted_User'] != example_user_id)])
                
                labels = ['Correct Predictions (User)', 'Incorrect Predictions (User)', 'Correct Predictions (Other)', 'Incorrect Predictions (Other)']
                sizes = [correct_predictions, incorrect_predictions, other_correct_predictions, other_incorrect_predictions]
                colors = ['green', 'red', 'blue', 'orange']
                
                plt.figure(figsize=(8, 8))
                plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
                plt.title('Prediction Accuracy Distribution')
                plt.show()

            plot_pie_chart(results_df, example_user_id)

NameError: name 'model_names' is not defined

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results, columns=['Model', 'Sample Size', 'Negative Ratio', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'])

# Plot the results
plt.figure(figsize=(12, 6))
for model_name in results_df['Model'].unique():
    for neg_ratio in negative_sample_ratios:
        model_results = results_df[(results_df['Model'] == model_name) & (results_df['Negative Ratio'] == neg_ratio)]
        plt.plot(model_results['Sample Size'], model_results['Accuracy'], marker='o', label=f'{model_name} (Neg Ratio {neg_ratio})')
plt.title('Accuracy vs. Sample Size for Different Models and Negative Ratios')
plt.xlabel('Sample Size')
plt.ylabel('Accuracy')
plt.xscale('log')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot additional metrics
metrics = ['Precision', 'Recall', 'F1 Score', 'ROC AUC']
for metric in metrics:
    plt.figure(figsize=(12, 6))
    for model_name in results_df['Model'].unique():
        for neg_ratio in negative_sample_ratios:
            model_results = results_df[(results_df['Model'] == model_name) & (results_df['Negative Ratio'] == neg_ratio)]
            plt.plot(model_results['Sample Size'], model_results[metric], marker='o', label=f'{model_name} (Neg Ratio {neg_ratio})')
    plt.title(f'{metric} vs. Sample Size for Different Models and Negative Ratios')
    plt.xlabel('Sample Size')
    plt.ylabel(metric)
    plt.xscale('log')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Summary of results
summary_metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
summary_df = results_df.groupby('Model')[summary_metrics].mean().reset_index()

# Plot summary of results
plt.figure(figsize=(12, 6))
for metric in summary_metrics:
    plt.plot(summary_df['Model'], summary_df[metric], marker='o', label=metric)
plt.title('Summary of Model Performance')
plt.xlabel('Model')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Zusammenfassung der Evaluationsergebnisse
def summarize_evaluation_results(results_df, example_user_id):
    # Überprüfen Sie die Spaltennamen im DataFrame
    print("Spalten im DataFrame:", results_df.columns)
    
    total_samples = len(results_df)
    correct_predictions_user = len(results_df[(results_df['Actual User'] == example_user_id) & (results_df['Predicted User'] == example_user_id)])
    incorrect_predictions_user = len(results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] == example_user_id)])
    correct_predictions_other = len(results_df[(results_df['Actual User'] != example_user_id) & (results_df['Predicted User'] != example_user_id)])
    incorrect_predictions_other = len(results_df[(results_df['Actual User'] == example_user_id) & (results_df['Predicted User'] != example_user_id)])
    
    summary = {
        'Total Samples': total_samples,
        'Correct Predictions (User)': correct_predictions_user,
        'Incorrect Predictions (User)': incorrect_predictions_user,
        'Correct Predictions (Other)': correct_predictions_other,
        'Incorrect Predictions (Other)': incorrect_predictions_other
    }
    
    return summary

# Beispiel für die Verwendung der Funktion
summary = summarize_evaluation_results(results_df, example_user_id)
print(summary)

In [ ]:
import tensorflow as tf
from numba import cuda

def clear_gpu_memory():
    # Clear TensorFlow GPU memory
    tf.keras.backend.clear_session()
    # Clear Numba GPU memory for both GPUs
    cuda.select_device(0)
    cuda.close()
    cuda.select_device(1)
    cuda.close()

# Beispiel für die Verwendung der Funktion
clear_gpu_memory()

def clear_gpu_memory():
    # Clear TensorFlow GPU memory
    tf.keras.backend.clear_session()
    # Clear Numba GPU memory for both GPUs
    cuda.select_device(0)
    cuda.close()
    cuda.select_device(1)
    cuda.close()

# Beispiel für die Verwendung der Funktion
clear_gpu_memory()